# Ingestion Pipeline

## 1. Transcript fetching

In [1]:
# fetch youtube trancript using url
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled,NoTranscriptFound

In [2]:
def youtube_id(url: str) -> str:
    '''
    youtube url format 

        http://www.youtube.com/watch?v=0zM3nApSvMg&feature=feedrec_grec_index
        http://www.youtube.com/user/IngridMichaelsonVEVO#p/a/u/1/QdK8U-VIH_o
        http://www.youtube.com/v/0zM3nApSvMg?fs=1&amp;hl=en_US&amp;rel=0
        http://www.youtube.com/watch?v=0zM3nApSvMg#t=0m10s
        http://www.youtube.com/embed/0zM3nApSvMg?rel=0
        http://www.youtube.com/watch?v=0zM3nApSvMg
        http://youtu.be/0zM3nApSvMg

    '''

    if "v=" in url:
        video_id = url.split("v=")[1].split("&")[0]
    elif "youtu.be/" in url:
        video_id = url.split("youtu.be/")[1]
    else:
        raise ValueError(f"video id not found for youtube video: {url}")
    
    return video_id
        


In [3]:
youtube_id("https://www.youtube.com/watch?v=ktrIQUYIxZo")

'ktrIQUYIxZo'

In [4]:
def gettranscript(url:str) -> list[dict]:
    video_id = youtube_id(url)
    try:
        api = YouTubeTranscriptApi()
        transcript = api.fetch(video_id=video_id)
        return transcript
    except TranscriptsDisabled:
        raise TranscriptsDisabled(
            f"transcript for this video is disabled, {video_id}"
        )
    except NoTranscriptFound:
        raise NoTranscriptFound(
            f"transcript for this video not found, {video_id}"
        )
        

In [5]:
transcript=gettranscript("https://www.youtube.com/watch?v=ktrIQUYIxZo")


In [6]:
transcript

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='Every day all I hear is how bad the data', start=0.24, duration=6.639), FetchedTranscriptSnippet(text='and machine learning job market is. And', start=3.919, duration=5.441), FetchedTranscriptSnippet(text="to be honest, I'm getting pretty sick of", start=6.879, duration=5.521), FetchedTranscriptSnippet(text='it. Over the past 5 years, I have', start=9.36, duration=5.92), FetchedTranscriptSnippet(text='consistently landed multiple 100k plus', start=12.4, duration=5.68), FetchedTranscriptSnippet(text='job offers both as a data scientist and', start=15.28, duration=4.8), FetchedTranscriptSnippet(text='machine learning engineer from big tech', start=18.08, duration=4.48), FetchedTranscriptSnippet(text="companies to startups. And [music] I've", start=20.08, duration=5.119), FetchedTranscriptSnippet(text='coached many people to do the exact same', start=22.56, duration=4.799), FetchedTranscriptSnippet(text='in my coaching program. Bu

In [7]:
transcript[-1].start

1100.32

In [8]:
def format_timestamp(seconds:float) -> str:
    seconds = int(seconds)
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    secs = seconds % 60 

    if hours > 0 :
        return {f"{hours}:{minutes:02d}:{secs:02d}"}
    else:
        return {f"{minutes:02d}:{secs:02d}"}

In [9]:
format_timestamp(transcript[-1].start)

{'18:20'}

In [10]:
len(transcript)

480

In [11]:
#top 5 segments with timestamp
for i , segment in enumerate(transcript[:5]):
    timestamp = format_timestamp(segment.start)
    print(f"{segment.text}, {timestamp}")

Every day all I hear is how bad the data, {'00:00'}
and machine learning job market is. And, {'00:03'}
to be honest, I'm getting pretty sick of, {'00:06'}
it. Over the past 5 years, I have, {'00:09'}
consistently landed multiple 100k plus, {'00:12'}


In [12]:
# last 5 segments with timestamp
for i , segment in enumerate(transcript[-5:]):
    timestamp = format_timestamp(segment.start)
    print(f"{segment.text}, {timestamp}")

work directly with me to land your dream, {'18:11'}
data or machine learning job within 3 to, {'18:13'}
6 months. There's a link below where you, {'18:16'}
can apply and I can't wait to work with, {'18:18'}
you. See you in the next video., {'18:20'}


## 2.Time Base Chunker 

In [13]:
def chunking_transcript(
        transcript : list[dict],
        video_id : str,
        window_sec : int=60,
        overlaping_sec : int=15
) -> list[dict]:
    
    chunks = []
    chunk_number = 0

    start_time = transcript[0].start
    end_time = transcript[-1].start + transcript[-1].duration

    step_sec = window_sec - overlaping_sec

    window_start = start_time

    while window_start < end_time:

        window_end = window_start + window_sec

        window_segments = [
            seg for seg in transcript
            if seg.start>=window_start and seg.start<=window_end
        ] 

        if not window_segments:
            window_start += step_sec
            continue

        combined_text = " ".join(
            seg.text.strip() for seg in window_segments
        )

        actual_start = window_segments[0].start
        last_seg = window_segments[-1]
        actual_end = last_seg.start + last_seg.duration

        chunk = {
            "chunk_id": f"{video_id}_chunk_{chunk_number}",
            "text" : combined_text,
            "start_time" : format_timestamp(actual_start),
            "end_time" : format_timestamp(actual_end),
            "start_sec" : actual_start,
            "video_id" :video_id
        }

        chunks.append(chunk)
        chunk_number+=1

        window_start += step_sec

    return chunks


In [15]:
chunks = chunking_transcript(transcript=transcript,video_id=youtube_id("https://www.youtube.com/watch?v=ktrIQUYIxZo"))

In [16]:
len(chunks)

25

In [17]:
chunks

[{'chunk_id': 'ktrIQUYIxZo_chunk_0',
  'text': "Every day all I hear is how bad the data and machine learning job market is. And to be honest, I'm getting pretty sick of it. Over the past 5 years, I have consistently landed multiple 100k plus job offers both as a data scientist and machine learning engineer from big tech companies to startups. And [music] I've coached many people to do the exact same in my coaching program. But [music] when I go online, apparently there are no data jobs available. So in this video, I want to explain and show you exactly what the state of the current job market is and go through [music] the three main things that you need to do to get hired. And make sure you stick around right to the end because the last one is a game changer. Let's get into it. Let me quickly spit some facts about what the actual job market looks like and explain to you exactly why it's not actually cooked as everyone online says",
  'start_time': {'00:00'},
  'end_time': {'01:04'},
 

## 3.Embedding chunks

In [21]:
from sentence_transformers import SentenceTransformer
import numpy as np
import time

In [19]:
embedding_model = SentenceTransformer("all-mpnet-base-v2")

e:\RAG_YT_assistant\rag-assistant\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in E:\AI_SETUP\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2179.49it/s]


In [20]:
vector_test = embedding_model.encode("hello gaurang lets build the RAG system")
vector_test

array([ 8.50838423e-03,  5.75877950e-02,  1.36363776e-02, -5.88553399e-03,
        5.37444465e-02, -1.67974792e-02, -5.42768510e-03, -5.52642718e-03,
       -3.00751738e-02, -8.82321037e-03,  7.60158570e-03, -3.56737971e-02,
       -3.11719514e-02,  6.10843934e-02,  5.96813038e-02, -4.44478467e-02,
       -7.91656878e-03,  1.50351680e-03, -5.87301478e-02, -2.20520627e-02,
        2.94128992e-02,  1.52085517e-02,  1.69245470e-02,  5.06254798e-03,
       -3.14935222e-02,  2.66195508e-03, -1.06549188e-02,  1.78795867e-02,
       -3.27539705e-02, -3.88478190e-02, -1.81164965e-02, -6.81349495e-03,
        1.56937353e-02,  5.13844043e-02,  1.88979266e-06, -3.38488221e-02,
        7.40575138e-03, -8.69232681e-05, -7.39119351e-02,  6.18941337e-02,
        6.26966506e-02,  1.85326785e-02, -3.34935412e-02,  2.89153438e-02,
       -2.71164649e-03, -8.60928968e-02,  1.57671552e-02, -1.42930066e-02,
        5.12605719e-03, -4.58069937e-03, -6.79122563e-03, -1.79014001e-02,
       -2.38534715e-02, -

In [27]:
def embed_chunks(
        chunks : list[dict],
        embedding_model : SentenceTransformer,
        batch_size : int=32
) -> list[dict]:
    start_time = time.time()
    texts = [chunk['text'] for chunk in chunks]

    vectors = embedding_model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True
    )
    embeddings = []
    for chunk,vector in zip(chunks,vectors):
        embeddings_chunks = {
            **chunk,
            "embedding" : vector.tolist()
        }
        embeddings.append(embeddings_chunks)

    finished_time = time.time() - start_time

    return embeddings        




In [29]:
embeded_chunks = embed_chunks(chunks=chunks,embedding_model=embedding_model)
embeded_chunks[:3]

Batches: 100%|██████████| 1/1 [00:04<00:00,  4.54s/it]


[{'chunk_id': 'ktrIQUYIxZo_chunk_0',
  'text': "Every day all I hear is how bad the data and machine learning job market is. And to be honest, I'm getting pretty sick of it. Over the past 5 years, I have consistently landed multiple 100k plus job offers both as a data scientist and machine learning engineer from big tech companies to startups. And [music] I've coached many people to do the exact same in my coaching program. But [music] when I go online, apparently there are no data jobs available. So in this video, I want to explain and show you exactly what the state of the current job market is and go through [music] the three main things that you need to do to get hired. And make sure you stick around right to the end because the last one is a game changer. Let's get into it. Let me quickly spit some facts about what the actual job market looks like and explain to you exactly why it's not actually cooked as everyone online says",
  'start_time': {'00:00'},
  'end_time': {'01:04'},
 

In [31]:
len(embeded_chunks[0]["embedding"])

768

## 4. Vector store

In [32]:
import chromadb
from chromadb.config import Settings

In [43]:
def get_vector_store(
        embedding_chunk : list[dict],
        video_id : str,
        persist_dir : str= "./chroma_db"
)-> chromadb.Collection:
    client = chromadb.PersistentClient(path=persist_dir)

    collection_name = f"yt_{video_id}"

    existing = [c.name for c in client.list_collections()]
    if collection_name in existing:
        client.delete_collection(collection_name)
    
    collections = client.create_collection(
        name  = collection_name,
        metadata= {"hnsw:space": "cosine"}
    )

    ids = [chunk["chunk_id"] for chunk in embedding_chunk]
    documents = [chunk["text"] for chunk in embedding_chunk]
    embeddings = [chunk["embedding"] for chunk in embedding_chunk]

    metadatas = [
        {
            "start_time" : str(chunk['start_time']),
            "end_time" : str(chunk["end_time"]),
            "start_sec" : float(chunk['start_sec']),
            "video_id" : str(chunk["video_id"])
        } for chunk in embedding_chunk
    ]

    collections.add(
        ids=ids,
        embeddings=embeddings,
        documents=documents,
        metadatas=metadatas,
    )

    return collections

In [44]:
get_vector_store(embedding_chunk=embeded_chunks,video_id=youtube_id("https://www.youtube.com/watch?v=ktrIQUYIxZo"))

Collection(name=yt_ktrIQUYIxZo)